In [2]:
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
import statsmodels.api as sm

In [3]:
df = pd.read_excel('augmented_survey.xlsx')

df['gender']         = df['Q1. What is your gender?']
df['allowance']      = df['Q2. What is your approximate monthly allowance / pocket money?']
df['year']           = df['Q3. Which year of college are you in?']
df['venue']          = df['Q4. What type of venue did you go to?']
df['day']            = df['Q5. What day was the outing?']
df['intiated_plan']  = df['Q6. Who initiated / planned the outing?']
df['occasion']       = df['Q7. What was the occasion?']
df['group_size']     = df['Q8. How many people were in your group, including yourself?']
df['travel_dist']    = df['Q9. How far did you travel to reach the venue?']
df['spending']       = df['Q10. Approximately how much did YOU personally spend on this outing? (₹) (Include food, travel, entry, shopping — everything)']
df['rough_budget']   = df['Q11. Did you have a rough budget in mind before going?']
df['exceed_budget']  = df['Q12. Did your actual spending exceed that budget? (Answer only if Q11 = Yes)']
df['spend_reason']   = df['Q13. What was the biggest reason you spent what you did? (Pick one)']
df['outing_freq']    = df['Q14. How often do you go on social outings per month?']
df['peer_influence'] = df["Q15. On a scale of 1–5, how much do your friends' choices influence where you go and what you spend? (1 = Not at all, 5 = Completely driven by friends)"]
df['discount']       = df['Q16. Did you use any discount or offer? (Zomato, Swiggy Dineout, student discount, etc.)']

df = df.drop(columns={'Q1. What is your gender?',
       'Q2. What is your approximate monthly allowance / pocket money?',
       'Q3. Which year of college are you in?',
       'Q4. What type of venue did you go to?', 'Q5. What day was the outing?',
       'Q6. Who initiated / planned the outing?', 'Q7. What was the occasion?',
       'Q8. How many people were in your group, including yourself?',
       'Q9. How far did you travel to reach the venue?',
       'Q10. Approximately how much did YOU personally spend on this outing? (₹) (Include food, travel, entry, shopping — everything)',
       'Q11. Did you have a rough budget in mind before going?',
       'Q12. Did your actual spending exceed that budget? (Answer only if Q11 = Yes)',
       'Q13. What was the biggest reason you spent what you did? (Pick one)',
       'Q14. How often do you go on social outings per month?',
       "Q15. On a scale of 1–5, how much do your friends' choices influence where you go and what you spend? (1 = Not at all, 5 = Completely driven by friends)",
       'Q16. Did you use any discount or offer? (Zomato, Swiggy Dineout, student discount, etc.)'})
df

,gender,allowance,year,venue,day,intiated_plan,occasion,group_size,travel_dist,spending,rough_budget,exceed_budget,spend_reason,outing_freq,peer_influence,discount
0,Female,"Less than ₹3,000",2nd Year,Café / Coffee Shop,Weekday (Mon–Thu),A friend did,Just hanging out (no specific reason),3–4 people,Less than 2 km,₹200 – ₹500,No,"Yes, I spent more than I planned",I was in a good mood,2–3 times,4,No
1,Female,"₹10,001 – ₹15,000",2nd Year,Movie Theatre,Weekday (Mon–Thu),I did,Just hanging out (no specific reason),5–7 people,2–5 km,"₹501 – ₹1,000",Yes,"Yes, I spent more than I planned",I was in a good mood,4–5 times,3,Yes
2,Female,"₹3,000 – ₹6,000",2nd Year,Café / Coffee Shop,Weekend (Sat–Sun),It was a group decision,Just hanging out (no specific reason),5–7 people,6–10 km,"₹501 – ₹1,000",Yes,"No, I stayed within my budget",The venue/place itself was expensive,2–3 times,3,No
3,Male,"₹6,001 – ₹10,000",1st Year,Pub / Bar / Lounge,Weekday (Mon–Thu),I did,After exams / stress relief,2 people,Less than 2 km,"₹501 – ₹1,000",Yes,"No, I stayed within my budget",I just didn't track / wasn't paying attention,2–3 times,3,No
4,Male,"₹3,000 – ₹6,000",1st Year,Park / Outdoor space,Weekday (Mon–Thu),"It was spontaneous, no one planned it",Just hanging out (no specific reason),3–4 people,2–5 km,Less than ₹200,No,"No, I stayed within my budget",I just didn't track / wasn't paying attention,More than 5 times,4,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,Male,"₹6,001 – ₹10,000",1st Year,Park / Outdoor space,Weekend (Sat–Sun),"It was spontaneous, no one planned it",Date / romantic outing,5–7 people,Less than 2 km,₹200 – ₹500,No,"Yes, I spent more than I planned",I was in a good mood,4–5 times,2,Yes
416,Female,"₹6,001 – ₹10,000",2nd Year,Pub / Bar / Lounge,Weekend (Sat–Sun),It was a group decision,"Celebration (birthday, farewell, etc.)",3–4 people,More than 10 km,"₹1,001 – ₹2,000",Yes,"No, I stayed within my budget",The venue/place itself was expensive,2–3 times,2,No
417,Male,"More than ₹15,000",2nd Year,Café / Coffee Shop,Weekend (Sat–Sun),I did,Just hanging out (no specific reason),8 or more,Less than 2 km,"₹501 – ₹1,000",No,"No, I stayed within my budget",I just didn't track / wasn't paying attention,More than 5 times,2,No
418,Male,"₹3,000 – ₹6,000",2nd Year,Pub / Bar / Lounge,Weekend (Sat–Sun),"It was spontaneous, no one planned it",Just hanging out (no specific reason),2 people,2–5 km,"₹501 – ₹1,000",Yes,I spent exactly what I planned,I was trying to save / was on a budget,Once or less,2,No


In [4]:
spend_map = {
    'Less than ₹200': 100,
    '₹200 – ₹500': 350,
    '₹501 – ₹1,000': 750,
    '₹1,001 – ₹2,000': 1500,
    'More than ₹2,000': 2500
}
allowance_map = {
    'Less than ₹3,000': 1500,
    '₹3,000 – ₹6,000': 4500,
    '₹6,001 – ₹10,000': 8000,
    '₹10,001 – ₹15,000': 12500,
    'More than ₹15,000': 17500
}

df['spending']  = df['spending'].map(spend_map)
df['allowance'] = df['allowance'].map(allowance_map)

In [5]:
df.columns

Index(['gender', 'allowance', 'year', 'venue', 'day', 'intiated_plan',
       'occasion', 'group_size', 'travel_dist', 'spending', 'rough_budget',
       'exceed_budget', 'spend_reason', 'outing_freq', 'peer_influence',
       'discount'],
      dtype='object')

In [6]:
df_real = df.iloc[:140]

In [7]:
fit1 = ols('spending ~ C(gender) + allowance + C(year) + C(venue) + C(day) + C(intiated_plan) + C(occasion) + C(group_size) + C(travel_dist) + C(rough_budget) + C(exceed_budget) + C(spend_reason) + C(outing_freq) + peer_influence + C(discount)', data=df_real).fit()
anova1 = sm.stats.anova_lm(fit1, typ=1)
anova1

,df,sum_sq,mean_sq,F,PR(>F)
C(gender),1.0,5.103288e+04,5.103288e+04,0.131249,0.717913
C(year),3.0,7.688306e+05,2.562769e+05,0.659105,0.579147
C(venue),6.0,1.740055e+06,2.900092e+05,0.745859,0.614083
C(day),2.0,4.631588e+04,2.315794e+04,0.059559,0.942214
C(intiated_plan),3.0,9.042218e+05,3.014073e+05,0.775173,0.510576
C(occasion),4.0,4.911598e+06,1.227899e+06,3.157969,0.017300
C(group_size),3.0,4.470084e+06,1.490028e+06,3.832123,0.012107
C(travel_dist),3.0,4.249209e+06,1.416403e+06,3.642771,0.015329
C(rough_budget),1.0,1.064719e+06,1.064719e+06,2.738293,0.101136
C(exceed_budget),3.0,3.559029e+05,1.186343e+05,0.305109,0.821637


## Spending affected by travel_dist, group_size, occasion

In [8]:
fit1 = ols('allowance ~ C(gender) + spending + C(year) + C(venue) + C(day) + C(intiated_plan) + C(occasion) + C(group_size) + C(travel_dist) + C(rough_budget) + C(exceed_budget) + C(spend_reason) + C(outing_freq) + peer_influence + C(discount)', data=df_real).fit()
anova1 = sm.stats.anova_lm(fit1, typ=1)
anova1

,df,sum_sq,mean_sq,F,PR(>F)
C(gender),1.0,1.583989e+07,1.583989e+07,0.758376,0.385944
C(year),3.0,1.944294e+08,6.480982e+07,3.102940,0.030083
C(venue),6.0,1.524806e+08,2.541343e+07,1.216735,0.304140
C(day),2.0,1.728768e+07,8.643839e+06,0.413846,0.662241
C(intiated_plan),3.0,1.312743e+08,4.375810e+07,2.095034,0.105724
C(occasion),4.0,2.414934e+08,6.037335e+07,2.890532,0.026072
C(group_size),3.0,5.575964e+06,1.858655e+06,0.088988,0.965940
C(travel_dist),3.0,1.388536e+08,4.628452e+07,2.215993,0.090994
C(rough_budget),1.0,8.381234e+07,8.381234e+07,4.012735,0.047891
C(exceed_budget),3.0,2.064117e+07,6.880391e+06,0.329417,0.804085


## allowance is affected by year, occasion, rough_budget

# One way ANOVA

### Q4: Does allowance differ across college years?

In [82]:
fit4 = ols('allowance ~ C(year)', data=df_real).fit()
anova4 = sm.stats.anova_lm(fit4, typ=1)
anova4

,df,sum_sq,mean_sq,F,PR(>F)
C(year),3.0,2.081928e+08,6.939761e+07,2.964647,0.034378
Residual,136.0,3.183541e+09,2.340839e+07,NaN,NaN


In [84]:
tukey = pairwise_tukeyhsd(df_real['allowance'], groups = df_real['year'])
tukey._results_table

group1,group2,meandiff,p-adj,lower,upper,reject
1st Year,2nd Year,2659.965,0.0705,-149.1138,5469.0439,False
1st Year,3rd Year,1807.6923,0.6902,-2467.0848,6082.4694,False
1st Year,4th Year / Final Year,4384.6154,0.0421,109.8383,8659.3925,True
2nd Year,3rd Year,-852.2727,0.9341,-4591.5459,2887.0004,False
2nd Year,4th Year / Final Year,1724.6503,0.6281,-2014.6228,5463.9235,False
3rd Year,4th Year / Final Year,2576.9231,0.5279,-2359.1643,7513.0105,False


## 4th year spends more than 1st year

### Q7: Does spending differ by travel distance?

In [70]:
fit4 = ols('spending ~ C(travel_dist)', data=df_real).fit()
anova4 = sm.stats.anova_lm(fit4, typ=1)
anova4

,df,sum_sq,mean_sq,F,PR(>F)
C(travel_dist),3.0,5.942666e+06,1.980889e+06,4.900466,0.002891
Residual,136.0,5.497455e+07,4.042246e+05,NaN,NaN


In [86]:
tukey = pairwise_tukeyhsd(df_real['spending'], groups = df_real['travel_dist'])
tukey._results_table

group1,group2,meandiff,p-adj,lower,upper,reject
2–5 km,6–10 km,247.5888,0.2763,-109.5364,604.714,False
2–5 km,Less than 2 km,-218.6275,0.4171,-588.0834,150.8285,False
2–5 km,More than 10 km,376.1094,0.1282,-68.3706,820.5894,False
6–10 km,Less than 2 km,-466.2162,0.0139,-862.1808,-70.2517,True
6–10 km,More than 10 km,128.5206,0.8905,-338.2263,595.2676,False
Less than 2 km,More than 10 km,594.7368,0.0079,118.489,1070.9847,True


## 10km spends most and 6-10 km spends second most

### Q9: Does allowance differ by discount usage?

In [72]:
fit3 = ols('allowance ~ C(discount)', data=df_real).fit()
anova3 = sm.stats.anova_lm(fit3, typ=1)
anova3

,df,sum_sq,mean_sq,F,PR(>F)
C(discount),1.0,1.086053e+08,1.086053e+08,4.565013,0.034396
Residual,138.0,3.283129e+09,2.379079e+07,NaN,NaN


In [93]:
tukey = pairwise_tukeyhsd(df_real['allowance'], groups = df_real['discount'])
tukey._results_table

group1,group2,meandiff,p-adj,lower,upper,reject
No,Yes,1762.2549,0.0344,131.3784,3393.1314,True


## people who use discounts spend more

In [101]:
# Spending ~ Occasion
tukey1 = pairwise_tukeyhsd(df_real['spending'], df_real['occasion'])
tukey1._results_table

group1,group2,meandiff,p-adj,lower,upper,reject
After exams / stress relief,"Celebration (birthday, farewell, etc.)",-64.2045,0.9991,-761.7581,633.349,False
After exams / stress relief,Date / romantic outing,551.25,0.2164,-166.6744,1269.1744,False
After exams / stress relief,Just hanging out (no specific reason),-76.75,0.992,-556.2858,402.7858,False
After exams / stress relief,Other,-685.4167,0.4427,-1805.9072,435.0739,False
"Celebration (birthday, farewell, etc.)",Date / romantic outing,615.4545,0.191,-162.699,1393.6081,False
"Celebration (birthday, farewell, etc.)",Just hanging out (no specific reason),-12.5455,1.0,-578.2858,553.1949,False
"Celebration (birthday, farewell, etc.)",Other,-621.2121,0.5767,-1781.215,538.7907,False
Date / romantic outing,Just hanging out (no specific reason),-628.0,0.0311,-1218.6747,-37.3253,True
Date / romantic outing,Other,-1236.6667,0.0332,-2409.0323,-64.3011,True
Just hanging out (no specific reason),Other,-608.6667,0.4919,-1652.2088,434.8755,False


## Most spending on dates/romantic outing

In [102]:
# Spending ~ Group Size
tukey2 = pairwise_tukeyhsd(df_real['spending'], df_real['group_size'])
tukey2._results_table

group1,group2,meandiff,p-adj,lower,upper,reject
2 people,3–4 people,-195.4072,0.4889,-553.2404,162.426,False
2 people,5–7 people,-78.2977,0.9605,-491.4404,334.8451,False
2 people,8 or more,465.0735,0.268,-199.2802,1129.4272,False
3–4 people,5–7 people,117.1096,0.8339,-244.2658,478.4849,False
3–4 people,8 or more,660.4808,0.0374,27.0211,1293.9404,True
5–7 people,8 or more,543.3712,0.1515,-122.8971,1209.6395,False


## 8 people spend most

# Two-way ANOVA

In [14]:
fit1 = ols('spending ~ C(travel_dist) * C(group_size) * C(occasion)', data=df_real).fit()
anova1 = sm.stats.anova_lm(fit1, typ=1)
anova1

,df,sum_sq,mean_sq,F,PR(>F)
C(travel_dist),3.0,5.942666e+06,1.980889e+06,5.599307,0.001381
C(group_size),3.0,2.054566e+06,6.848553e+05,1.935856,0.128788
C(occasion),4.0,4.785715e+06,1.196429e+06,3.381901,0.012300
C(travel_dist):C(group_size),9.0,4.579881e+06,5.088756e+05,1.438420,0.182295
C(travel_dist):C(occasion),12.0,5.336373e+06,4.446978e+05,1.257011,0.256575
C(group_size):C(occasion),12.0,4.610025e+06,3.841687e+05,1.085916,0.380396
C(travel_dist):C(group_size):C(occasion),36.0,1.960179e+07,5.444942e+05,1.539102,0.049598
Residual,98.0,3.466985e+07,3.537740e+05,NaN,NaN


In [18]:
tukey = pairwise_tukeyhsd(df_real['spending'], df_real['group_size']+df_real['travel_dist']+df_real['occasion'])
tukey_df = pd.DataFrame(tukey._results_table.data[1:], columns=tukey._results_table.data[0])
true_values = tukey_df[tukey_df['reject'] == True]
true_values

,group1,group2,meandiff,p-adj,lower,upper,reject


In [19]:
fit1 = ols('allowance ~ C(year) * C(rough_budget) * C(occasion)', data=df_real).fit()
anova1 = sm.stats.anova_lm(fit1, typ=1)
anova1

,df,sum_sq,mean_sq,F,PR(>F)
C(year),3.0,2.081928e+08,6.939761e+07,3.301465,0.022891
C(rough_budget),1.0,6.492999e+07,6.492999e+07,3.088926,0.081488
C(occasion),4.0,3.219316e+08,8.048290e+07,3.828827,0.005856
C(year):C(rough_budget),3.0,9.196283e+07,3.065428e+07,1.458321,0.229652
C(year):C(occasion),12.0,1.910851e+08,1.592376e+07,0.757544,0.692356
C(rough_budget):C(occasion),4.0,1.068196e+08,2.670489e+07,1.270437,0.285704
C(year):C(rough_budget):C(occasion),12.0,4.103969e+08,3.419974e+07,1.626991,0.093489
Residual,115.0,2.417328e+09,2.102025e+07,NaN,NaN
